# Fine-tune DistilBERT for Intent Classification

This notebook fine-tunes distilbert-base-uncased on the Bitext customer support dataset for intent classification.

Uses the same dataset split as baseline_models.ipynb

## Setup and Dependencies

In [20]:
# Install required libraries
!pip install datasets transformers scikit-learn torch wandb -q --upgrade

In [21]:
from google.colab import userdata

WANDB_API_KEY = userdata.get('WANDB_API_KEY')

In [22]:
import os
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

import wandb

In [23]:
# Initialize wandb
wandb.init(
    project="intent-classification",
    name="distilbert-finetune",
    config={
        "model_name": "distilbert-base-uncased",
        "epochs": 3,
        "learning_rate": 2e-5,
        "batch_size": 16,
        "max_length": 128,
    }
)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Using device: cuda
GPU: Tesla T4


## Load and Prepare Dataset

In [24]:
# Load the dataset
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
print(f"Dataset splits: {dataset.keys()}")
print(f"Train set size: {len(dataset['train'])}")

Dataset splits: dict_keys(['train'])
Train set size: 26872


In [25]:
# Use same split
dataset['train'] = dataset['train'].shuffle(seed=42)

train_testvalid = dataset['train'].train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

split_dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']
})

print(f"Train set size: {len(split_dataset['train'])}")
print(f"Validation set size: {len(split_dataset['validation'])}")
print(f"Test set size: {len(split_dataset['test'])}")

Train set size: 18810
Validation set size: 4031
Test set size: 4031


In [26]:
# Get label information
labels = split_dataset['train']['intent']
unique_labels = sorted(set(labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Number of classes: {len(unique_labels)}")
print(f"Classes: {unique_labels}")

Number of classes: 27
Classes: ['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


## Tokenization

In [27]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 128

def tokenize_function(examples):
    tokens = tokenizer(
        examples['instruction'],
        padding='max_length',
        truncation=True,
        max_length=max_length
    )
    tokens['labels'] = [label2id[label] for label in examples['intent']]
    return tokens

# Tokenize all splits
print("Tokenizing dataset...")
tokenized_dataset = split_dataset.map(tokenize_function, batched=True, remove_columns=['instruction', 'intent', 'response', 'category', 'flags'])
print("Tokenization complete")

Tokenizing dataset...


Map:   0%|          | 0/4031 [00:00<?, ? examples/s]

Tokenization complete


## Training Setup

In [28]:
# Load model
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded with {len(unique_labels)} labels")

Loading model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded with 27 labels


In [29]:
# Define compute_metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'f1': macro_f1
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 6,
    report_to="wandb",
    save_total_limit=1,
)

print("Training arguments configured")
print(f"Using fp16: {training_args.fp16}")

## Train the Model

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

print("Trainer created, starting training...")

In [ ]:
# Train
train_result = trainer.train()

print(f"\nTraining completed!")
print(f"Final training loss: {train_result.training_loss:.4f}")

## Evaluate on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(eval_dataset=tokenized_dataset['test'])

print(f"\nTest Results:")
print(f"Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Macro F1: {test_results['eval_f1']:.4f}")

In [ ]:
# Get predictions on test set for detailed analysis
import torch
from tqdm import tqdm

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(trainer.get_test_dataloader(), desc="Predicting"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**{k: v for k, v in batch.items() if k != 'labels'})
        predictions = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("\nDetailed Classification Report:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=unique_labels,
    digits=4
))

## Log Metrics to W&B

In [ ]:
# Calculate per-class F1 scores
per_class_f1 = f1_score(all_labels, all_preds, average=None)

# Log final metrics
wandb.log({
    "test_accuracy": test_results['eval_accuracy'],
    "test_macro_f1": test_results['eval_f1'],
    **{f"test_f1_{label}": f1 for label, f1 in zip(unique_labels, per_class_f1)}
})

print("Metrics logged to W&B")

## Save Model

In [ ]:
# Save model and tokenizer
model_save_path = './distilbert-intent-classifier'
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")

# Also save label mappings
import json
with open(f'{model_save_path}/label_mapping.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

print("Label mappings saved")

In [ ]:
# Finish wandb run
wandb.finish()